# TaFi Video Studio — Kaggle GPU Edition
Chạy toàn bộ pipeline trên **GPU miễn phí của Kaggle** (T4/P100 x2, ~30 giờ/tuần):
- ASR: **SenseVoice** (FunASR chạy CUDA)
- OCR: **PP-OCRv5 server** + CUDA
- Render: **h264_nvenc** (GPU) — video 2 tiếng chỉ ~5–10 phút
- Giữ nguyên: dịch xKiro + QC chính tả + CapCut TTS + dictionary gate

**Cách dùng:** tạo notebook ở kaggle.com/code (New Notebook) → **File ▸ Upload Notebook**, chọn file này.
Sau đó bật **Settings ▸ Accelerator = GPU T4 x2 / P100** và **Internet = ON** (bắt buộc), rồi **Run all**.
Chạy xong copy link `https://xxx.trycloudflare.com` ở cell 7 để mở web.

⚠ Kaggle tắt phiên sau ~9–12h hoặc ~90 phút không hoạt động → tải file output (`/kaggle/working/tafi/jobs/`) về trước khi hết.


In [ ]:
# 1) Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'Không có GPU — vẫn chạy được nhưng chậm hơn'
import torch
print('torch CUDA:', torch.cuda.is_available())


In [ ]:
# 2) Lấy code (tự clone repo GitHub chứa sẵn code + zip; nếu clone lỗi sẽ tự tải zip)
import os, zipfile, glob, shutil
os.makedirs('/kaggle/working', exist_ok=True)
GIT_URL = 'https://github.com/theandanh000-cmyk/tafi-colab-gpu.git'
os.system(f"cd /kaggle/working && git clone --depth 1 {GIT_URL} tafi 2>/dev/null || true")
if not os.path.exists('/kaggle/working/tafi/server.js'):
    print('Clone lỗi → tải zip từ GitHub...')
    os.system("cd /kaggle/working && rm -f tafi.zip && curl -sL -o tafi.zip https://github.com/theandanh000-cmyk/tafi-colab-gpu/raw/main/TaFi-VS-Tool-Web.zip")
    os.system("cd /kaggle/working && rm -rf tafi && mkdir tafi && cd tafi && unzip -q -o ../tafi.zip && rm -f ../tafi.zip")
    if not os.path.exists('/kaggle/working/tafi/server.js'):
        inner = glob.glob('/kaggle/working/tafi/*/server.js')
        if inner:
            src = os.path.dirname(inner[0])
            os.system(f"rm -rf /kaggle/working/tafi && cp -r '{src}' /kaggle/working/tafi")
os.chdir('/kaggle/working/tafi')
print('Code tại:', os.getcwd())
print('server.js:', os.path.exists('server.js'))


In [ ]:
# 3) Cài môi trường (GPU)
!apt-get -qq update >/dev/null 2>&1 && apt-get -qq install -y ffmpeg unzip >/dev/null 2>&1
!pip -q install numpy pillow opencv-python-headless rapidocr opencc-python-reimplemented spylls you-get curl_cffi yt-dlp sherpa-onnx
# GPU cho OCR (thay onnxruntime bằng bản GPU)
!pip -q install --upgrade onnxruntime-gpu
# ASR GPU bằng FunASR SenseVoice (Kaggle đã có sẵn torch CUDA)
!pip -q install funasr modelscope
!cd /kaggle/working/tafi && (cd pipeline/gemini_translator && npm install --omit=dev 2>&1 | tail -1)
print('Đã cài xong deps')


In [ ]:
# 4) Tải model ASR SenseVoice (sherpa int8, dùng khi đổi ASR_ENGINE=sensevoice)
!mkdir -p /kaggle/working/tafi/pipeline/models
!cd /kaggle/working/tafi/pipeline/models && { test -f sherpa-onnx-sense-voice-zh-en-ja-ko-yue-int8-2024-07-17/model.int8.onnx || { curl -sL -o sv.tar.bz2 https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-sense-voice-zh-en-ja-ko-yue-int8-2024-07-17.tar.bz2 && tar xjf sv.tar.bz2 && rm -f sv.tar.bz2; }; }
print('Model sherpa sensevoice sẵn sàng')


In [ ]:
# 5) Khởi động server với cấu hình GPU (dịch vẫn xKiro — GPU chỉ dùng ASR/OCR/render)
import os, subprocess, time
os.environ.update({
    'ASR_ENGINE': 'sensevoice-funasr',   # GPU thật (FunASR); đổi 'sensevoice' = sherpa CPU (nhanh, đỡ tài nguyên)
    'OCR_MODEL': 'ppocrv5-server',       # OCR chuẩn nhất, chạy CUDA
    'OCR_USE_CUDA': '1',
    'RENDER_CODEC': 'h264_nvenc',        # render GPU
    'PORT': '3000',
    'TRANSLATE_ENGINE': 'xkiro',         # dịch xKiro (free, không cần GPU)
})
os.system("cd /kaggle/working/tafi && pkill -f 'node server.js' 2>/dev/null; nohup node server.js > /tmp/tafi.log 2>&1 &")
time.sleep(8)
print(os.popen("curl -s localhost:3000/api/health | head -c 120").read())


In [ ]:
# 6) Mở web ra internet bằng tunnel cloudflared (cần Internet ON trong Settings)
!which cloudflared >/dev/null 2>&1 || (curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)
import subprocess, re
proc = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:3000','--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('🌐 LINK WEB (mở trên điện thoại/máy):', url)
open('/tmp/tunnel_url.txt','w').write(url or '')


In [ ]:
# 7) Giữ phiên sống (tự ping mỗi 60 phút) — Kaggle tắt nếu ~90 phút không hoạt động
import time, subprocess
for i in range(600):
    time.sleep(60)
    subprocess.run(['curl','-s','-o','/dev/null','localhost:3000/api/health'])
    print(f'giữ phiên {i+1}/600 phút — link web vẫn ở /tmp/tunnel_url.txt', flush=True)


## Ghi chú
- **Bắt buộc:** Settings ▸ Accelerator = GPU (T4 x2 hoặc P100), **Internet = ON**. Kaggle miễn phí ~30 giờ GPU/tuần.
- ASR mặc định `sensevoice-funasr` = FunASR + SenseVoiceSmall chạy CUDA. Nếu cài funasr lỗi, đổi `ASR_ENGINE=sensevoice` (sherpa-onnx, vẫn nhanh).
- OCR: `ppocrv5-server` tự tải lần đầu (~165MB) vào thư mục rapidocr; chỉ quét vùng đáy + bỏ khung lặp như bản local.
- Render: `h264_nvenc` tự dùng GPU; không có GPU thì đặt `RENDER_CODEC=libx264`.
- Key xKiro: đặt file `pipeline/gemini_translator/xkiro_key.txt` (hoặc env `XKIRO_API_KEY`) trước khi dịch.
- File output nằm trong `/kaggle/working/tafi/jobs/<jobId>/` — tải về máy trước khi hết phiên (Kaggle có nút download file).
